In [ ]:
!pip -q install rapidfuzz
import os, re, unicodedata, pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
if not os.path.exists('open/train.csv'):
    from google.colab import files; files.upload(); os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
t = pd.read_csv('open/train.csv'); c = pd.read_csv('open/vessel_catalog.csv')
k = t[t.vessel_name.str.contains('[가-힣]')].merge(c[['vessel_id','vessel_name','gross_tonnage']], left_on='target_vessel_id', right_on='vessel_id', suffixes=('','_en'))
U = k.drop_duplicates('target_vessel_id').reset_index(drop=True)   # 선박당 1행, 382척
MID = 'Qwen/Qwen2.5-3B-Instruct'
tk = AutoTokenizer.from_pretrained(MID); lm = AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.bfloat16).cuda().eval()
print(len(U), '척 준비')

In [ ]:
from rapidfuzz import fuzz, process
SHOT = U.sample(8, random_state=0); EV = U.drop(SHOT.index)   # 예시 8척은 평가에서 제외
msgs0 = [{'role': 'system', 'content': 'Convert a Korean (Hangul) transliteration of a ship name back to the original English/Latin ship name used in international ship registries. Answer with the name only.'}]
for a, b in zip(SHOT.vessel_name, SHOT.vessel_name_en):
    msgs0 += [{'role': 'user', 'content': a}, {'role': 'assistant', 'content': b}]

@torch.no_grad()
def back(kor, n=4):  # 한 행씩 생성 (행 독립)
    enc = tk.apply_chat_template(msgs0 + [{'role': 'user', 'content': kor}], add_generation_prompt=True,
                                 return_tensors='pt', return_dict=True).to('cuda')
    out = lm.generate(**enc, max_new_tokens=24, num_beams=n, num_return_sequences=n, do_sample=False)
    L = enc['input_ids'].shape[1]
    return [tk.decode(o[L:], skip_special_tokens=True).strip() for o in out]

norm = lambda s: re.sub(r'[^A-Z0-9]', '', unicodedata.normalize('NFKC', str(s)).upper())
CN = [norm(x) for x in c.vessel_name]; GT = c.gross_tonnage.values; IDS = c.vessel_id.values
res = []
for r in EV.itertuples():
    gens = back(r.vessel_name)
    S = np.max(process.cdist([norm(g) for g in gens], CN, scorer=fuzz.ratio, workers=-1), axis=0)  # 생성 후보 중 최고 유사도
    gap = np.abs(GT - r.gross_tonnage) / np.maximum(GT, 1); gap = np.where(np.isfinite(gap), gap, 9)
    res.append(dict(kor=r.vessel_name, true=r.vessel_name_en, gen=gens[0],
                    name_only=IDS[np.argmax(S)] == r.target_vessel_id,
                    with_gt=IDS[np.argmax(np.where(gap <= .35, S - 30 * gap, -1e9))] == r.target_vessel_id))
R = pd.DataFrame(res)
print('LLM 이름만:', R.name_only.mean().round(3), '| LLM + 톤수:', R.with_gt.mean().round(3))
print(R.head(20).to_string())

In [ ]:
trv = set(t.target_vessel_id)
pool = c[~c.vessel_id.isin(trv) & c.gross_tonnage.notna()]
smp = pd.concat([pool.sample(1200, random_state=0),
                 pool[pool.flag_name == 'South Korea'].sample(400, random_state=0)]).drop_duplicates('vessel_id')
msgsK = [{'role': 'system', 'content': 'Write how this ship name would appear in a Korean port record, transliterated into Hangul. Keep digits as digits. Answer with the Hangul name only.'}]
for a, b in zip(SHOT.vessel_name_en, SHOT.vessel_name):
    msgsK += [{'role': 'user', 'content': a}, {'role': 'assistant', 'content': b}]

@torch.no_grad()
def to_kor(eng):
    enc = tk.apply_chat_template(msgsK + [{'role': 'user', 'content': eng}], add_generation_prompt=True,
                                 return_tensors='pt', return_dict=True).to('cuda')
    out = lm.generate(**enc, max_new_tokens=32, do_sample=False)
    return tk.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

from tqdm.auto import tqdm
smp['kor_gen'] = [to_kor(e) for e in tqdm(smp.vessel_name)]
smp[['vessel_id', 'vessel_name', 'kor_gen']].to_csv('llm_kor_names.csv', index=False)
print(smp[['vessel_name', 'vessel_type', 'kor_gen']].head(25).to_string())
files.download('llm_kor_names.csv')

In [ ]:
!pip -q install rapidfuzz
import os, pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files
if not os.path.exists('open/train.csv'):
    files.upload(); os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
t = pd.read_csv('open/train.csv'); c = pd.read_csv('open/vessel_catalog.csv')
k = t[t.vessel_name.str.contains('[가-힣]')].merge(c[['vessel_id','vessel_name']], left_on='target_vessel_id', right_on='vessel_id', suffixes=('','_en')).drop_duplicates('target_vessel_id')
SHOT = k.sample(12, random_state=1)
MID = 'Qwen/Qwen2.5-7B-Instruct'
tk = AutoTokenizer.from_pretrained(MID, padding_side='left')
lm = AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.bfloat16).cuda().eval()
msgs = [{'role': 'system', 'content': 'You write ship names as they appear in Korean port records: the English name transliterated into Hangul syllables only (no Latin letters), digits kept as digits. Answer with the Hangul name only.'}]
for a, b in zip(SHOT.vessel_name_en, SHOT.vessel_name):
    msgs += [{'role': 'user', 'content': a}, {'role': 'assistant', 'content': b}]
pool = c[~c.vessel_id.isin(set(t.target_vessel_id)) & c.gross_tonnage.notna()]
smp = pd.concat([pool.sample(5000, random_state=7), pool[pool.flag_name == 'South Korea'].sample(1000, random_state=7)]).drop_duplicates('vessel_id')
prompts = [tk.apply_chat_template(msgs + [{'role': 'user', 'content': e}], add_generation_prompt=True, tokenize=False) for e in smp.vessel_name]
outs = []
from tqdm.auto import tqdm
with torch.no_grad():
    for i in tqdm(range(0, len(prompts), 48)):
        enc = tk(prompts[i:i+48], return_tensors='pt', padding=True).to('cuda')
        g = lm.generate(**enc, max_new_tokens=32, do_sample=False)
        outs += [s.strip() for s in tk.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
smp['kor_gen'] = outs
smp[['vessel_id', 'vessel_name', 'kor_gen']].to_csv('llm_kor_names_7b.csv', index=False)
print(smp[['vessel_name', 'vessel_type', 'kor_gen']].head(25).to_string())
files.download('llm_kor_names_7b.csv')

In [ ]:
!pip -q install rapidfuzz
import os, json, pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files
if not os.path.exists('open/train.csv'):
    files.upload(); os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
up = files.upload()  # kor_names_for_llm.csv
t = pd.read_csv('open/train.csv'); c = pd.read_csv('open/vessel_catalog.csv')
k = t[t.vessel_name.str.contains('[가-힣]')].merge(c[['vessel_id','vessel_name']], left_on='target_vessel_id', right_on='vessel_id', suffixes=('','_en')).drop_duplicates('target_vessel_id')
SHOT = k.sample(10, random_state=3)
shots = list(zip(SHOT.vessel_name, SHOT.vessel_name_en))
json.dump(shots, open('llm_shots.json', 'w'), ensure_ascii=False)   # 제출 코드에서도 똑같이 사용

MID = 'Qwen/Qwen2.5-3B-Instruct'
tk = AutoTokenizer.from_pretrained(MID, padding_side='left')
lm = AutoModelForCausalLM.from_pretrained(MID, dtype=torch.bfloat16).cuda().eval()
SYS = 'Convert a Korean (Hangul) transliteration of a ship name back to the original English/Latin ship name used in international ship registries. Answer with the name only.'
def prompt(kor):
    m = [{'role': 'system', 'content': SYS}]
    for a, b in shots: m += [{'role': 'user', 'content': a}, {'role': 'assistant', 'content': b}]
    return tk.apply_chat_template(m + [{'role': 'user', 'content': kor}], add_generation_prompt=True, tokenize=False)

N = pd.read_csv('kor_names_for_llm.csv').kor.astype(str).tolist(); G = []
from tqdm.auto import tqdm
with torch.no_grad():
    for i in tqdm(range(0, len(N), 32)):
        enc = tk([prompt(x) for x in N[i:i+32]], return_tensors='pt', padding=True).to('cuda')
        g = lm.generate(**enc, max_new_tokens=24, num_beams=4, num_return_sequences=4, do_sample=False)
        d = [s.strip().split('\n')[0] for s in tk.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
        G += [d[j*4:(j+1)*4] for j in range(len(d)//4)]
out = pd.DataFrame(G, columns=['g1','g2','g3','g4']); out.insert(0, 'kor', N)
out.to_csv('llm_back.csv', index=False); print(out.head(15).to_string())
lm.save_pretrained('qwen3b'); tk.save_pretrained('qwen3b')   # 제출 파일용으로 보관
files.download('llm_back.csv'); files.download('llm_shots.json')

In [ ]:
import os, glob, shutil, time
up = files.upload()  # submit_v10_code.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
del lm; import gc; gc.collect(); torch.cuda.empty_cache()   # 셀 E에서 올린 모델 메모리 비우기
!rm -rf sub && mkdir sub && unzip -oq submit_v10_code.zip -d sub
shutil.copytree('qwen3b', 'sub/model/qwen3b')
!pip -q install lightgbm
!rm -rf /content/tf457 && pip -q install --no-deps --target /content/tf457 transformers==4.57.6 huggingface-hub==0.36.2 tokenizers==0.22.1
!mkdir -p sub/data && cp open/vessel_catalog.csv sub/data/
tt = pd.read_csv('open/train.csv'); kk = tt.vessel_name.str.contains('[가-힣]')
s = pd.concat([tt[kk].sample(100, random_state=1), tt[~kk].sample(200, random_state=1)])   # 한글 100행 포함
s.drop(columns='target_vessel_id').to_csv('sub/data/test.csv', index=False)
t0 = time.time()
!cd sub && PYTHONPATH=/content/tf457 python script.py 2>&1 | grep -v -i warn | tail -3
print('총', round(time.time() - t0), '초 / 한글 100행')

In [ ]:
import json
for d in ['qwen3b', 'sub/model/qwen3b']:
    p = f'{d}/tokenizer_config.json'; cfg = json.load(open(p))
    est = cfg.pop('extra_special_tokens', None)
    if isinstance(est, dict): cfg['extra_special_tokens'] = est   # 예전 형식이면 유지
    json.dump(cfg, open(p, 'w'), ensure_ascii=False, indent=2)
    print(d, '수정 완료, 제거한 값:', type(est).__name__)
!cd sub && PYTHONPATH=/content/tf457 python -c "import sys; sys.path.insert(0,'model'); from llm_back import Back; b=Back('model/qwen3b','model/llm_shots.json'); print(b('모닝쿠리어'))"

In [ ]:
!rm -rf sub/output
t0 = time.time()
!cd sub && PYTHONPATH=/content/tf457 python script.py 2>&1 | grep -v -i warn | grep -v I0000 | tail -3
print('총', round(time.time() - t0), '초 / 한글 100행 + 비한글 200행')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf sub/data sub/output && find sub -name __pycache__ -prune -exec rm -rf {} \;
!cd sub && zip -r -0 -q /content/submit_v10.zip model script.py requirements.txt
!ls -la /content/submit_v10.zip && cp /content/submit_v10.zip /content/drive/MyDrive/
print('드라이브 저장 완료')

In [ ]:
import os, glob, pandas as pd, torch, sys
from google.colab import files
up = files.upload()  # s2_pairs.csv, llm_ll_code.zip
for f in glob.glob('* (*).*'): os.replace(f, f.split(' (')[0] + os.path.splitext(f)[1])
!unzip -oq llm_ll_code.zip -d llmcode
sys.path.insert(0, 'llmcode'); from llm_back import Back
path = 'qwen3b' if os.path.exists('qwen3b') else 'Qwen/Qwen2.5-3B-Instruct'
B = Back(path, 'llmcode/llm_shots.json'); print('model:', path, B.dev)
print(B.ll(['Morning Courier', 'Morning Lily', 'Ever Reliance'], '모닝쿠리어'))   # 첫 번째가 가장 높아야 정상

S = pd.read_csv('s2_pairs.csv'); out = []
from tqdm.auto import tqdm
for kor, g in tqdm(S.groupby('kor', sort=False)):   # 한 한글 이름의 후보들 = 한 배치 (제출 때와 동일)
    out += list(zip([kor] * len(g), g.cand_name, B.ll(g.cand_name.astype(str).tolist(), kor)))
R = pd.DataFrame(out, columns=['kor', 'cand_name', 'qll']); R.to_csv('s2_ll.csv', index=False)
print(R.head(10)); files.download('s2_ll.csv')

In [ ]:
import gc, importlib
up = files.upload()  # 새 llm_ll_code.zip
for f in glob.glob('* (*).*'): os.replace(f, f.split(' (')[0] + os.path.splitext(f)[1])
!unzip -oq llm_ll_code.zip -d llmcode
import llm_back; importlib.reload(llm_back)
lm_, tk_ = B.lm, B.tk; del B; gc.collect(); torch.cuda.empty_cache()
B = llm_back.Back.__new__(llm_back.Back); B.lm, B.tk, B.dev = lm_, tk_, 'cuda'
B.shots = json.load(open('llmcode/llm_shots.json', encoding='utf-8')) if 'json' in dir() else __import__('json').load(open('llmcode/llm_shots.json', encoding='utf-8'))
print(B.ll(['Morning Courier', 'Morning Lily', 'Ever Reliance'], '모닝쿠리어'))   # 이전 값(-6.0, -24.5, -50.6)과 거의 같아야 정상

S = pd.read_csv('s2_pairs.csv'); out = []
from tqdm.auto import tqdm
for kor, g in tqdm(S.groupby('kor', sort=False)):
    out += list(zip([kor] * len(g), g.cand_name, B.ll(g.cand_name.astype(str).tolist(), kor)))
R = pd.DataFrame(out, columns=['kor', 'cand_name', 'qll']); R.to_csv('s2_ll.csv', index=False)
print(R.head(10)); files.download('s2_ll.csv')

In [ ]:
import gc, importlib, json
up = files.upload()  # 새 llm_ll_code.zip
for f in glob.glob('* (*).*'): os.replace(f, f.split(' (')[0] + os.path.splitext(f)[1])
!unzip -oq llm_ll_code.zip -d llmcode
importlib.reload(llm_back)
B2 = llm_back.Back.__new__(llm_back.Back); B2.lm, B2.tk, B2.dev = B.lm, B.tk, 'cuda'
B2.shots = json.load(open('llmcode/llm_shots.json', encoding='utf-8'))
print(B2.ll(['Morning Courier', 'Morning Lily', 'Ever Reliance'], '모닝쿠리어'))   # 첫 번째가 가장 높으면 OK (예시 수가 바뀌어 값은 조금 달라짐)
import time; t0 = time.time()
for _ in range(5): B2.ll([f'Cand {i}' for i in range(20)], '모닝쿠리어')
print('후보 20개 1회:', round((time.time() - t0) / 5, 3), '초')

In [ ]:
up = files.upload()  # s2_pairs10.csv
S = pd.read_csv('s2_pairs10.csv'); out = []
from tqdm.auto import tqdm
for kor, g in tqdm(S.groupby('kor', sort=False)):   # 한 한글 이름의 후보들 = 한 배치 (제출 때와 동일)
    out += list(zip([kor] * len(g), g.cand_name, B2.ll(g.cand_name.astype(str).tolist(), kor)))
R = pd.DataFrame(out, columns=['kor', 'cand_name', 'qll']); R.to_csv('s2_ll.csv', index=False)
print(R.head(10)); files.download('s2_ll.csv')

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, glob, time, pandas as pd
!rm -rf sub && mkdir sub && cd sub && unzip -oq /content/drive/MyDrive/submit_v10.zip
up = files.upload()  # submit_v11_code.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!cd sub && unzip -oq ../submit_v11_code.zip && head -1 script.py && ls model | head -30
!pip -q install rapidfuzz lightgbm
!rm -rf /content/tf457 && pip -q install --no-deps --target /content/tf457 transformers==4.57.6 huggingface-hub==0.36.2 tokenizers==0.22.1
# 리허설: 한글 100행 + 비한글 200행
if not os.path.exists('open/train.csv'):
    files.upload(); os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
!mkdir -p sub/data && cp open/vessel_catalog.csv sub/data/
tt = pd.read_csv('open/train.csv'); kk = tt.vessel_name.str.contains('[가-힣]')
pd.concat([tt[kk].sample(100, random_state=1), tt[~kk].sample(200, random_state=1)]).drop(columns='target_vessel_id').to_csv('sub/data/test.csv', index=False)
t0 = time.time()
!cd sub && PYTHONPATH=/content/tf457 python script.py 2>&1 | grep -v -i warn | grep -v I0000 | tail -3
print('총', round(time.time() - t0), '초')

In [ ]:
!rm -rf sub/data sub/output && find sub -name __pycache__ -prune -exec rm -rf {} \;
!cd sub && zip -r -0 -q /content/drive/MyDrive/submit_v11.zip model script.py requirements.txt && ls -la /content/drive/MyDrive/submit_v11.zip

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, glob
print('v10 있음:', os.path.exists('/content/drive/MyDrive/submit_v10.zip'))   # True여야 함
!rm -rf sub && mkdir sub && cd sub && unzip -oq /content/drive/MyDrive/submit_v10.zip
up = files.upload()  # submit_v11_code.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!cd sub && unzip -oq ../submit_v11_code.zip && head -1 script.py
!find sub -name __pycache__ -prune -exec rm -rf {} \;
# 1) 먼저 코랩 디스크에 만들고  2) 드라이브로 복사  3) 크기 비교
!cd sub && zip -r -0 -q /content/submit_v11.zip model script.py requirements.txt
!cp /content/submit_v11.zip /content/drive/MyDrive/submit_v11.zip
a = os.path.getsize('/content/submit_v11.zip'); b = os.path.getsize('/content/drive/MyDrive/submit_v11.zip')
print('코랩:', a, '/ 드라이브:', b, '/ 같음:', a == b)
drive.flush_and_unmount(); print('동기화 완료')

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, glob
!rm -rf sub && mkdir sub && cd sub && unzip -oq /content/drive/MyDrive/submit_v10.zip
up = files.upload()  # submit_v12_code.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!cd sub && rm -f model/kor_stage2.txt && unzip -oq ../submit_v12_code.zip && head -1 script.py
!find sub -name __pycache__ -prune -exec rm -rf {} \;
!cd sub && zip -r -0 -q /content/submit_v12.zip model script.py requirements.txt && ls -la /content/submit_v12.zip

In [ ]:
# ===== LoRA 미세조정 (Qwen2.5-3B, 한글 선박명 -> 영문) : 2-fold + 전체 =====
!pip -q install peft rapidfuzz
import os, re, json, gc, time, unicodedata, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from rapidfuzz import fuzz, process
from google.colab import files
need = [f for f in ['ft_pairs.csv', 'gen_names.csv'] if not os.path.exists(f)]
if not os.path.exists('open/vessel_catalog.csv') and not os.path.exists('open_data.zip'): need.append('open_data.zip')
if need: print('올려주세요:', need); files.upload()
if not os.path.exists('open/vessel_catalog.csv'): os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
P = pd.read_csv('ft_pairs.csv'); N = pd.read_csv('gen_names.csv'); c = pd.read_csv('open/vessel_catalog.csv')
MID = 'Qwen/Qwen2.5-3B-Instruct'
tk = AutoTokenizer.from_pretrained(MID, padding_side='left')
SYS = 'Convert a Korean (Hangul) transliteration of a ship name back to the original English/Latin ship name used in international ship registries. Answer with the name only.'
prompt = lambda kor: tk.apply_chat_template([{'role': 'system', 'content': SYS}, {'role': 'user', 'content': kor}], add_generation_prompt=True, tokenize=False)

def train(df, epochs=2, bs=16, lr=2e-4, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    m = AutoModelForCausalLM.from_pretrained(MID, dtype=torch.bfloat16).cuda()
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    ex = []
    for k, e in zip(df.kor.astype(str), df.eng.astype(str)):
        p = tk(prompt(k))['input_ids']; a = tk(e + '<|im_end|>')['input_ids']; ex.append((p + a, [-100] * len(p) + a))
    opt = torch.optim.AdamW([q for q in m.parameters() if q.requires_grad], lr=lr)
    steps = epochs * (len(ex) // bs); sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps)); m.train(); t0 = time.time()
    for ep in range(epochs):
        order = rng.permutation(len(ex))
        for i in range(0, len(order) - bs + 1, bs):
            b = [ex[j] for j in order[i:i + bs]]; L = max(len(x) for x, _ in b)
            ids = torch.tensor([x + [tk.pad_token_id] * (L - len(x)) for x, _ in b]).cuda()
            lab = torch.tensor([y + [-100] * (L - len(y)) for _, y in b]).cuda()
            loss = m(input_ids=ids, attention_mask=(torch.arange(L)[None] < torch.tensor([len(x) for x, _ in b])[:, None]).long().cuda(), labels=lab).loss
            loss.backward(); opt.step(); sch.step(); opt.zero_grad()
        print(f'  epoch {ep} loss {loss.item():.3f} ({time.time() - t0:.0f}s)')
    return m.merge_and_unload().eval()

@torch.no_grad()
def gen(m, names, bs=32):
    out = []
    for i in range(0, len(names), bs):
        enc = tk([prompt(x) for x in names[i:i + bs]], return_tensors='pt', padding=True).to('cuda')
        g = m.generate(**enc, max_new_tokens=24, num_beams=4, num_return_sequences=4, do_sample=False)
        d = [s.strip().split('\n')[0] for s in tk.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
        out += [d[j * 4:(j + 1) * 4] for j in range(len(d) // 4)]
    return out

norm = lambda s: re.sub(r'[^A-Z0-9]', '', unicodedata.normalize('NFKC', str(s)).upper())
CN = [norm(x) for x in c.vessel_name]; IDS = c.vessel_id.values
def name_only_acc(m, df):   # 학습에 안 쓴 실제 한글 이름 -> 역음역 -> 카탈로그 전체 이름만으로 1등
    G = gen(m, df.kor.astype(str).tolist()); ok = 0
    for gs, v in zip(G, df.vessel):
        S = np.max(process.cdist([norm(x) for x in gs], CN, scorer=fuzz.ratio, workers=-1), axis=0); ok += IDS[np.argmax(S)] == v
    return ok / len(df)

res = {}
for f in [0, 1]:
    print(f'== fold {f}: 학습 {len(P[P.fold != f])}쌍'); m = train(P[P.fold != f])
    ev = P[(P.fold == f) & (P.src == 'real')]; print('  실제 한글 이름만으로 정확도 (기존 few-shot 3B는 약 0.79):', round(name_only_acc(m, ev), 4))
    nm = N[N.fold == f].kor.astype(str).tolist(); print(f'  역음역 생성 {len(nm)}개...'); res.update(dict(zip(nm, gen(m, nm))))
    del m; gc.collect(); torch.cuda.empty_cache()
out = pd.DataFrame([[k] + v for k, v in res.items()], columns=['kor', 'g1', 'g2', 'g3', 'g4']); out.to_csv('llm_back_ft.csv', index=False)
print(out.head(10).to_string()); files.download('llm_back_ft.csv')

print('== 최종 모델: 전체로 학습'); m = train(P)
m.save_pretrained('qwen3b_ft'); tk.save_pretrained('qwen3b_ft')
p = 'qwen3b_ft/tokenizer_config.json'; cfg = json.load(open(p)); est = cfg.pop('extra_special_tokens', None)
if isinstance(est, dict): cfg['extra_special_tokens'] = est
json.dump(cfg, open(p, 'w'), ensure_ascii=False, indent=2); print('qwen3b_ft 저장 완료 (서버 버전용 설정 수정 포함)')

In [ ]:
!pip uninstall -y torchao

In [ ]:
import os, glob, time, pandas as pd
from google.colab import files
print('qwen3b_ft 있음:', os.path.exists('qwen3b_ft'))
up = files.upload()  # submit_v14_code.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!rm -rf sub && mkdir sub && cd sub && unzip -oq ../submit_v14_code.zip && head -1 script.py
!cp -r qwen3b_ft sub/model/qwen3b_ft
!pip -q install lightgbm rapidfuzz
!rm -rf /content/tf457 && pip -q install --no-deps --target /content/tf457 transformers==4.57.6 huggingface-hub==0.36.2 tokenizers==0.22.1
!mkdir -p sub/data && cp open/vessel_catalog.csv sub/data/
tt = pd.read_csv('open/train.csv'); kk = tt.vessel_name.str.contains('[가-힣]')
pd.concat([tt[kk].sample(100, random_state=1), tt[~kk].sample(200, random_state=1)]).drop(columns='target_vessel_id').to_csv('sub/data/test.csv', index=False)
!cd sub && PYTHONPATH=/content/tf457 python -c "import sys; sys.path.insert(0,'model'); from llm_back import Back; print(Back('model/qwen3b_ft','model/llm_shots.json')('모닝쿠리어'))"
t0 = time.time()
!cd sub && PYTHONPATH=/content/tf457 python script.py 2>&1 | grep -v -i warn | grep -v I0000 | tail -3
print('총', round(time.time() - t0), '초')
!rm -rf sub/data sub/output && find sub -name __pycache__ -prune -exec rm -rf {} \;
!cd sub && zip -r -0 -q /content/submit_v14.zip model script.py requirements.txt && ls -la /content/submit_v14.zip

In [ ]:
import gc, torch
for v in ['m', 'model', 'lm']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()
print('GPU 사용 중:', round(torch.cuda.memory_allocated() / 1e9, 1), 'GB')   # 0 근처여야 함
!mkdir -p sub/data && cp open/vessel_catalog.csv sub/data/
tt = pd.read_csv('open/train.csv'); kk = tt.vessel_name.str.contains('[가-힣]')
pd.concat([tt[kk].sample(100, random_state=1), tt[~kk].sample(200, random_state=1)]).drop(columns='target_vessel_id').to_csv('sub/data/test.csv', index=False)
!cd sub && PYTHONPATH=/content/tf457 python -c "import sys; sys.path.insert(0,'model'); from llm_back import Back; print(Back('model/qwen3b_ft','model/llm_shots.json')('모닝쿠리어'))" 2>&1 | tail -1
t0 = time.time()
!cd sub && PYTHONPATH=/content/tf457 python script.py

In [ ]:
import os, glob, shutil
from google.colab import files
up = files.upload()  # v14a_code.zip ~ v14e_code.zip 5개 한 번에 선택
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
for v in 'abcde':
    !rm -rf ov_{v} && mkdir ov_{v} && unzip -oq v14{v}_code.zip -d ov_{v}
    shutil.copy('/content/submit_v14.zip', f'/content/submit_v14{v}.zip')               # 6GB 복사
    !cd ov_{v} && zip -0 -q /content/submit_v14{v}.zip $(find . -type f | sed 's#^\./##')  # 바뀐 파일만 교체
    !unzip -p /content/submit_v14{v}.zip script.py | head -1
!ls -la /content/submit_v14*.zip

In [ ]:
# ===== LoRA 2차 (epoch 4, r=32, 한글 오타·약칭 증강) : 2-fold + 전체 =====
!pip -q install peft rapidfuzz
import os, re, json, gc, time, unicodedata, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from rapidfuzz import fuzz, process
from google.colab import files
for v in ['m', 'model', 'lm', 'B', 'B2']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()
need = [f for f in ['ft_pairs.csv', 'gen_names.csv'] if not os.path.exists(f)]
if not os.path.exists('open/vessel_catalog.csv') and not os.path.exists('open_data.zip'): need.append('open_data.zip')
if need: print('올려주세요:', need); files.upload()
if not os.path.exists('open/vessel_catalog.csv'): os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
P = pd.read_csv('ft_pairs.csv'); N = pd.read_csv('gen_names.csv'); c = pd.read_csv('open/vessel_catalog.csv')
MID = 'Qwen/Qwen2.5-3B-Instruct'
tk = AutoTokenizer.from_pretrained(MID, padding_side='left')
SYS = 'Convert a Korean (Hangul) transliteration of a ship name back to the original English/Latin ship name used in international ship registries. Answer with the name only.'
prompt = lambda kor: tk.apply_chat_template([{'role': 'system', 'content': SYS}, {'role': 'user', 'content': kor}], add_generation_prompt=True, tokenize=False)

VSWAP = {1: 5, 5: 1, 4: 8, 8: 4, 13: 18, 18: 13, 20: 18}   # ㅐ↔ㅔ, ㅓ↔ㅗ, ㅜ↔ㅡ, ㅣ→ㅡ
def ktypo(s, rng):
    ch = list(s); idx = [i for i, x in enumerate(ch) if 0 <= ord(x) - 0xAC00 < 11172]
    if not idx: return s
    i = idx[rng.randint(len(idx))]; op = rng.choice(['vowel', 'del', 'dup', 'space']); o = ord(ch[i]) - 0xAC00
    cho, jung, jong = o // 588, (o % 588) // 28, o % 28
    if op == 'vowel' and jung in VSWAP: ch[i] = chr(0xAC00 + cho * 588 + VSWAP[jung] * 28 + jong)
    elif op == 'del' and len(idx) > 2: del ch[i]
    elif op == 'dup': ch.insert(i, ch[i])
    elif op == 'space': ch.insert(i, ' ')
    return ''.join(ch)
def kabbrev(s, rng):
    w = s.split()
    if len(w) < 2: return s.replace(' ', '')
    return ''.join(w) if rng.rand() < .5 else ' '.join(w[:-1])
def augment(df, seed=1):   # 한글 쪽에 오타·붙여쓰기 변형을 넣은 쌍을 같은 수만큼 추가
    rng = np.random.RandomState(seed); rows = []
    for k, e in zip(df.kor.astype(str), df.eng.astype(str)):
        k2 = ktypo(k, rng) if rng.rand() < .7 else kabbrev(k, rng)
        if k2 != k: rows.append((k2, e))
    return pd.concat([df[['kor', 'eng']], pd.DataFrame(rows, columns=['kor', 'eng'])], ignore_index=True)

def train(df, epochs=4, bs=16, lr=2e-4, seed=0):
    df = augment(df, seed + 1); print(f'  증강 후 {len(df)}쌍')
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    m = AutoModelForCausalLM.from_pretrained(MID, dtype=torch.bfloat16).cuda()
    m = get_peft_model(m, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type='CAUSAL_LM',
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    ex = []
    for k, e in zip(df.kor.astype(str), df.eng.astype(str)):
        p = tk(prompt(k))['input_ids']; a = tk(e + '<|im_end|>')['input_ids']; ex.append((p + a, [-100] * len(p) + a))
    opt = torch.optim.AdamW([q for q in m.parameters() if q.requires_grad], lr=lr)
    steps = epochs * (len(ex) // bs); sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps)); m.train(); t0 = time.time()
    for ep in range(epochs):
        order = rng.permutation(len(ex))
        for i in range(0, len(order) - bs + 1, bs):
            b = [ex[j] for j in order[i:i + bs]]; L = max(len(x) for x, _ in b)
            ids = torch.tensor([x + [tk.pad_token_id] * (L - len(x)) for x, _ in b]).cuda()
            lab = torch.tensor([y + [-100] * (L - len(y)) for _, y in b]).cuda()
            loss = m(input_ids=ids, attention_mask=(torch.arange(L)[None] < torch.tensor([len(x) for x, _ in b])[:, None]).long().cuda(), labels=lab).loss
            loss.backward(); opt.step(); sch.step(); opt.zero_grad()
        print(f'  epoch {ep} loss {loss.item():.3f} ({time.time() - t0:.0f}s)')
    return m.merge_and_unload().eval()

@torch.no_grad()
def gen(m, names, bs=32):
    out = []
    for i in range(0, len(names), bs):
        enc = tk([prompt(x) for x in names[i:i + bs]], return_tensors='pt', padding=True).to('cuda')
        g = m.generate(**enc, max_new_tokens=24, num_beams=4, num_return_sequences=4, do_sample=False)
        d = [s.strip().split('\n')[0] for s in tk.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
        out += [d[j * 4:(j + 1) * 4] for j in range(len(d) // 4)]
    return out

norm = lambda s: re.sub(r'[^A-Z0-9]', '', unicodedata.normalize('NFKC', str(s)).upper())
CN = [norm(x) for x in c.vessel_name]; IDS = c.vessel_id.values
def name_only_acc(m, df):   # 학습에 안 쓴 실제 한글 이름 -> 역음역 -> 카탈로그 전체 이름만으로 1등
    G = gen(m, df.kor.astype(str).tolist()); ok = 0
    for gs, v in zip(G, df.vessel):
        S = np.max(process.cdist([norm(x) for x in gs], CN, scorer=fuzz.ratio, workers=-1), axis=0); ok += IDS[np.argmax(S)] == v
    return ok / len(df)

res = {}
for f in [0, 1]:
    print(f'== fold {f}: 학습 {len(P[P.fold != f])}쌍'); m = train(P[P.fold != f])
    ev = P[(P.fold == f) & (P.src == 'real')]; print('  실제 한글 이름만으로 정확도:', round(name_only_acc(m, ev), 4))
    nm = N[N.fold == f].kor.astype(str).tolist(); print(f'  역음역 생성 {len(nm)}개...'); res.update(dict(zip(nm, gen(m, nm))))
    del m; gc.collect(); torch.cuda.empty_cache()
out = pd.DataFrame([[k] + v for k, v in res.items()], columns=['kor', 'g1', 'g2', 'g3', 'g4']); out.to_csv('llm_back_ft2.csv', index=False)
print(out.head(10).to_string()); files.download('llm_back_ft2.csv')

print('== 최종 모델: 전체로 학습'); m = train(P)
m.save_pretrained('qwen3b_ft2'); tk.save_pretrained('qwen3b_ft2')
p = 'qwen3b_ft2/tokenizer_config.json'; cfg = json.load(open(p)); est = cfg.pop('extra_special_tokens', None)
if isinstance(est, dict): cfg['extra_special_tokens'] = est
json.dump(cfg, open(p, 'w'), ensure_ascii=False, indent=2); print('qwen3b_ft2 저장 완료 (서버 버전용 설정 수정 포함)')

In [ ]:
# ===== 7B QLoRA (Qwen2.5-7B, 4비트) : 한글 선박명 -> 영문, 2-fold + 전체 + 4비트 저장 =====
!pip -q install peft bitsandbytes rapidfuzz
!pip -q uninstall -y torchao
import os, re, json, gc, time, unicodedata, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from rapidfuzz import fuzz, process
from google.colab import files
for v in ['m', 'model', 'lm', 'B', 'B2']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()
need = [f for f in ['ft_pairs.csv', 'gen_names.csv'] if not os.path.exists(f)]
if not os.path.exists('open/vessel_catalog.csv') and not os.path.exists('open_data.zip'): need.append('open_data.zip')
if need: print('올려주세요:', need); files.upload()
if not os.path.exists('open/vessel_catalog.csv'): os.system('unzip -oq open_data.zip && unzip -oq open.zip -d open')
P = pd.read_csv('ft_pairs.csv'); N = pd.read_csv('gen_names.csv'); c = pd.read_csv('open/vessel_catalog.csv')

MID = 'Qwen/Qwen2.5-7B-Instruct'
Q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tk = AutoTokenizer.from_pretrained(MID, padding_side='left')
SYS = 'Convert a Korean (Hangul) transliteration of a ship name back to the original English/Latin ship name used in international ship registries. Answer with the name only.'
prompt = lambda kor: tk.apply_chat_template([{'role': 'system', 'content': SYS}, {'role': 'user', 'content': kor}], add_generation_prompt=True, tokenize=False)

def train(df, epochs=2, bs=16, lr=2e-4, seed=0):   # LoRA 1차(3B)에서 잘 됐던 설정 그대로
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    m = AutoModelForCausalLM.from_pretrained(MID, quantization_config=Q4, dtype=torch.bfloat16, device_map={'': 0})
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    ex = []
    for k, e in zip(df.kor.astype(str), df.eng.astype(str)):
        p = tk(prompt(k))['input_ids']; a = tk(e + '<|im_end|>')['input_ids']; ex.append((p + a, [-100] * len(p) + a))
    opt = torch.optim.AdamW([q for q in m.parameters() if q.requires_grad], lr=lr)
    steps = epochs * (len(ex) // bs); sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps)); m.train(); t0 = time.time()
    for ep in range(epochs):
        order = rng.permutation(len(ex))
        for i in range(0, len(order) - bs + 1, bs):
            b = [ex[j] for j in order[i:i + bs]]; L = max(len(x) for x, _ in b)
            ids = torch.tensor([x + [tk.pad_token_id] * (L - len(x)) for x, _ in b]).cuda()
            lab = torch.tensor([y + [-100] * (L - len(y)) for _, y in b]).cuda()
            am = (torch.arange(L)[None] < torch.tensor([len(x) for x, _ in b])[:, None]).long().cuda()
            loss = m(input_ids=ids, attention_mask=am, labels=lab).loss
            loss.backward(); opt.step(); sch.step(); opt.zero_grad()
        print(f'  epoch {ep} loss {loss.item():.3f} ({time.time() - t0:.0f}s)')
    m.eval(); m.config.use_cache = True
    return m

@torch.no_grad()
def gen(m, names, bs=32, beams=1):   # 제출 때는 greedy(후보 1개)
    out = []
    for i in range(0, len(names), bs):
        enc = tk([prompt(x) for x in names[i:i + bs]], return_tensors='pt', padding=True).to('cuda')
        g = m.generate(**enc, max_new_tokens=24, num_beams=beams, num_return_sequences=beams, do_sample=False)
        d = [s.strip().split('\n')[0] for s in tk.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
        out += [d[j * beams:(j + 1) * beams] for j in range(len(d) // beams)]
    return out

norm = lambda s: re.sub(r'[^A-Z0-9]', '', unicodedata.normalize('NFKC', str(s)).upper())
CN = [norm(x) for x in c.vessel_name]; IDS = c.vessel_id.values
def name_only_acc(m, df):
    G = gen(m, df.kor.astype(str).tolist()); ok = 0
    for gs, v in zip(G, df.vessel):
        S = np.max(process.cdist([norm(x) for x in gs], CN, scorer=fuzz.ratio, workers=-1), axis=0); ok += IDS[np.argmax(S)] == v
    return ok / len(df)

res = {}
for f in [0, 1]:
    print(f'== fold {f}: 학습 {len(P[P.fold != f])}쌍'); m = train(P[P.fold != f])
    ev = P[(P.fold == f) & (P.src == 'real')]; print('  실제 한글 이름만으로 정확도 (greedy):', round(name_only_acc(m, ev), 4))
    if f == 0:   # 제출 시간 가늠: 한 행씩 greedy
        t0 = time.time(); [gen(m, [x]) for x in ev.kor.astype(str).head(30)]; print('  한 행 생성 시간:', round((time.time() - t0) / 30, 3), '초')
    nm = N[N.fold == f].kor.astype(str).tolist(); print(f'  역음역 생성 {len(nm)}개...'); res.update(dict(zip(nm, gen(m, nm))))
    del m; gc.collect(); torch.cuda.empty_cache()
out = pd.DataFrame([[k] + v for k, v in res.items()], columns=['kor', 'g1']); out.to_csv('llm_back_q7.csv', index=False)
print(out.head(10).to_string()); files.download('llm_back_q7.csv')

print('== 최종 모델: 전체로 학습'); m = train(P); m.save_pretrained('q7_adapter'); del m; gc.collect(); torch.cuda.empty_cache()
# bf16 원본에 LoRA를 합친 뒤 4비트로 다시 저장 (제출용, 약 5.5GB)
base = AutoModelForCausalLM.from_pretrained(MID, dtype=torch.bfloat16, device_map={'': 0})
merged = PeftModel.from_pretrained(base, 'q7_adapter').merge_and_unload(); merged.save_pretrained('q7_merged_bf16'); tk.save_pretrained('q7_merged_bf16')
del base, merged; gc.collect(); torch.cuda.empty_cache()
q = AutoModelForCausalLM.from_pretrained('q7_merged_bf16', quantization_config=Q4, dtype=torch.bfloat16, device_map={'': 0})
q.save_pretrained('qwen7b_ft_q4'); tk.save_pretrained('qwen7b_ft_q4')
p = 'qwen7b_ft_q4/tokenizer_config.json'; cfg = json.load(open(p)); est = cfg.pop('extra_special_tokens', None)
if isinstance(est, dict): cfg['extra_special_tokens'] = est
json.dump(cfg, open(p, 'w'), ensure_ascii=False, indent=2)
!du -sh qwen7b_ft_q4 && rm -rf q7_merged_bf16
print('qwen7b_ft_q4 저장 완료')

In [ ]:
import os, glob, gc, time, json, torch, pandas as pd
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
for v in ['m', 'q', 'base', 'merged', 'model', 'lm']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()
print('3B:', os.path.exists('qwen3b_ft'), '/ 7B:', os.path.exists('qwen7b_ft_q4'))
# 1) 3B LoRA를 4비트로 저장
Q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
q = AutoModelForCausalLM.from_pretrained('qwen3b_ft', quantization_config=Q4, dtype=torch.bfloat16, device_map={'': 0})
q.save_pretrained('qwen3b_ft_q4'); AutoTokenizer.from_pretrained('qwen3b_ft').save_pretrained('qwen3b_ft_q4')
p = 'qwen3b_ft_q4/tokenizer_config.json'; cfg = json.load(open(p)); est = cfg.pop('extra_special_tokens', None)
if isinstance(est, dict): cfg['extra_special_tokens'] = est
json.dump(cfg, open(p, 'w'), ensure_ascii=False, indent=2)
del q; gc.collect(); torch.cuda.empty_cache()
!du -sh qwen3b_ft_q4 qwen7b_ft_q4
import bitsandbytes; print('bitsandbytes', bitsandbytes.__version__)
# 2) 조립
up = files.upload()  # submit_v15_code.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!rm -rf sub && mkdir sub && cd sub && unzip -oq ../submit_v15_code.zip && head -1 script.py && cat requirements.txt
!cp -r qwen3b_ft_q4 qwen7b_ft_q4 sub/model/
# 3) 리허설 (한글 100행 + 비한글 200행)
!pip -q install lightgbm rapidfuzz
!rm -rf /content/tf457 && pip -q install --no-deps --target /content/tf457 transformers==4.57.6 huggingface-hub==0.36.2 tokenizers==0.22.1
!mkdir -p sub/data && cp open/vessel_catalog.csv sub/data/
tt = pd.read_csv('open/train.csv'); kk = tt.vessel_name.str.contains('[가-힣]')
pd.concat([tt[kk].sample(100, random_state=1), tt[~kk].sample(200, random_state=1)]).drop(columns='target_vessel_id').to_csv('sub/data/test.csv', index=False)
!cd sub && PYTHONPATH=/content/tf457 python -c "import sys; sys.path.insert(0,'model'); from llm_back import Back; print(Back('model/qwen3b_ft_q4','model/llm_shots.json')('모닝쿠리어')); print(Back('model/qwen7b_ft_q4','model/llm_shots.json').greedy('모닝쿠리어'))" 2>&1 | tail -2
t0 = time.time()
!cd sub && PYTHONPATH=/content/tf457 python script.py 2>&1 | grep -v -i warn | grep -v I0000 | tail -3
print('총', round(time.time() - t0), '초 / 한글 100행 + 비한글 200행')
!rm -rf sub/data sub/output && find sub -name __pycache__ -prune -exec rm -rf {} \;
!cd sub && zip -r -0 -q /content/submit_v15.zip model script.py requirements.txt && ls -la /content/submit_v15.zip

In [ ]:
import os, glob, shutil
from google.colab import files
print('v15 있음:', os.path.exists('/content/submit_v15.zip'))
up = files.upload()  # v15s_overlay.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!rm -rf ov15s && mkdir ov15s && unzip -oq v15s_overlay.zip -d ov15s
shutil.copy('/content/submit_v15.zip', '/content/submit_v15s.zip')
!cd ov15s && zip -0 -q /content/submit_v15s.zip $(find . -type f | sed 's#^\./##')
!unzip -p /content/submit_v15s.zip script.py | head -1 && ls -la /content/submit_v15s.zip